In [0]:
from pyspark.sql.functions import *

In [0]:
import sys
import importlib
sys.path.append('/Workspace/Users/kiranmanam9490@outlook.com/customer360')

import common.configuration
importlib.reload(common.configuration)
common.configuration.dbutils = dbutils

from common.configuration import Configuration

conf = Configuration()

In [0]:
df = spark.read.format('parquet').load(conf.base_url + '/bronze/support_tickets')


In [0]:
df = df.withColumn(
    "ticket_date",
    coalesce(
        try_to_date(col("ticket_date"), "yyyy/MM/dd"),
        try_to_date(col("ticket_date"), "dd-MM-yyyy"),
        try_to_date(col("ticket_date"), "yyyy-MM-dd"),
        try_to_date(col("ticket_date"), "yyyyMMdd"),
        try_to_date(col("ticket_date"), "MM-dd-yyyy"),
        try_to_date(col("ticket_date"), "dd/MM/yyyy"),
    )
)

In [0]:

df = df.withColumn(
    "issue_type",
    initcap(
        when(
            lower(col("issue_type").cast("string")) == "na",
            "Unknown"
        ).otherwise(col("issue_type"))
    )
)

In [0]:
df =df.withColumn('resolution_status',initcap(col("resolution_status")))

In [0]:
df =df.dropDuplicates(["ticket_id","customer_id"])
df = df.dropna(subset=["ticket_id","customer_id"])

In [0]:
df = df.withColumn("customer_id", col("customer_id").cast("int"))

In [0]:
df.write.format("delta").mode("overwrite").save(conf.base_url + '/silver/support_tickets')